# UMUD — EDA + AutoMM baseline (free GPU)

**Setup before Run All:**
1. Notebook Settings → Accelerator: **GPU T4 x2** (or any GPU), Internet: **ON**
2. **Add Data → Competitions → UMUD Challenge** (mounts under `/kaggle/input`)

Pipeline: discover → manifests (+SHA-256 dedup) → EDA → splits → 3× AutoMM regressors → `submission.csv`.
If the organizers shipped **no numeric-label CSV**, derive targets from masks first (see the labels cell).

In [ ]:
!nvidia-smi -L
!git clone -b arena/01a0dbe5-discordbot https://github.com/Duytan1801/discordbot.git 2>&1 | tail -1
!pip install -q -r discordbot/umud/requirements-kaggle.txt 2>&1 | tail -2
import sys; sys.path.insert(0, 'discordbot/umud/src')
from pathlib import Path
_cands = [Path('/kaggle/working/discordbot/umud/src'), Path('discordbot/umud/src')]
SRC = str(next(p for p in _cands if (p / 'data.py').exists()))
print('SRC =', SRC)

In [ ]:
!ls /kaggle/input
!python $SRC/data.py --discover

In [ ]:
!python $SRC/data.py --manifests

In [ ]:
!python $SRC/eda.py

# Show the sample figures inline
from pathlib import Path
from IPython.display import Image, display
for png in sorted(Path('discordbot/umud/reports').glob('*.png')):
    print(png.name)
    display(Image(str(png), width=800))

## Labels decision point

Check the discover output: is there a CSV with `pa_deg/fl_mm/mt_mm` columns?
- **Yes →** point `--labels` at it in the splits cell below.
- **No →** derive targets from masks with `geometry.py`. First resolve **px→mm** (check tif tags / scale bars / dataset docs): PA needs no scale; FL/MT do. Start with `--px-per-mm 1.0`, then fit one global scale factor on validation if needed.

In [ ]:
# Option A: organizer labels exist
# !python $SRC/data.py --splits --labels /path/to/labels.csv

# Option B: derive from masks (uncomment, set px-per-mm!)
# !python $SRC/geometry.py --derive --manifest discordbot/umud/data/manifests/fasc_unique.csv \
#     --apo-manifest discordbot/umud/data/manifests/apo_unique.csv --px-per-mm 1.0 \
#     -o discordbot/umud/data/manifests/labels_from_masks.csv
# !python $SRC/data.py --splits --labels discordbot/umud/data/manifests/labels_from_masks.csv

In [ ]:
# Phase A baselines — one predictor per target (AutoMM is single-label).
# Bump --time-limit / --preset (→ best_quality) once the pipeline works.
!python $SRC/train_regression.py --target pa_deg --preset medium_quality --time-limit 600
!python $SRC/train_regression.py --target fl_mm  --preset medium_quality --time-limit 600
!python $SRC/train_regression.py --target mt_mm  --preset medium_quality --time-limit 600

In [ ]:
# Predict test + build submission.csv
from pathlib import Path
import pandas as pd
from autogluon.multimodal import MultiModalPredictor

test = pd.read_csv('discordbot/umud/data/manifests/test.csv')
runs = Path('discordbot/umud/runs')
for t in ['pa_deg', 'fl_mm', 'mt_mm']:
    run = sorted(runs.glob(f'{t}_*'))[-1]
    preds = MultiModalPredictor.load(str(run)).predict(test[['image']])
    pd.DataFrame({'image_id': test['image_id'], t: preds.to_numpy().ravel()}).to_csv(f'test_{t}.csv', index=False)
    print(t, 'from', run.name)

!python $SRC/submit.py --pa test_pa_deg.csv --fl test_fl_mm.csv --mt test_mt_mm.csv -o submission.csv
!head submission.csv

## Next (Phase B/C)

- `train_segmentation.py --task apo/fasc` → SAM masks → `geometry.measure()` → geometric submission; compare UMUD(val) vs regression.
- Scale regression: `--preset best_quality --backbone convnext_base_384`, longer time, `--ballpark`.
- `ensemble.py`: UMUD-tuned weights, `--groups` temporal smoothing on the 5-frame sequences, range clipping.
- Track everything by UMUD score on val (`metrics.umud_score`), never by raw MAE alone.